# Classical eight games

Estimate classical Elo from **eight games** by one player (10% / 50% / 90% band). Features are averaged across the eight games, then scored once — not eight one-game guesses averaged. The one-game trainer is `notebooks/train_site.ipynb`.

**Data:** Lichess `lichess_db_standard_rated_2017-02.pgn`, Elo 800–2400. See `docs/data.md`.

**Colab setup:**

- `MyDrive/DATA/player_games.parquet`
- `MyDrive/DATA/player_sample_meta.json`
- `MyDrive/src/` — full `src` package

Bundle writes to `MyDrive/DATA/player8`.

**How the search works:** median uses quantile pinball; early stopping uses macro per-bin MAE. `SEARCH_GRID` may vary depth, `min_child_weight`, or learning rate; lowest validation macro MAE wins (ties keep the frozen row). Overall MAE, within-150, and coverage are reported only. Interval scale is fit on validation; test is scored once.

**Ship when:** test macro MAE ≤ 120, tail bias within ±80 Elo, and every bin has typical 10–90% width ≤ 350 Elo with coverage ≥ 70%.


In [ ]:
from pathlib import Path
import sys

if not Path("/content").exists():
    raise RuntimeError(
        "This notebook trains in Google Colab. Open notebooks/train_player8.ipynb there "
        "and run every cell from the top."
    )

from google.colab import drive
import subprocess

drive.mount("/content/drive")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pyarrow>=14", "pandas>=2.0", "numpy>=1.24",
    "scikit-learn>=1.3", "scipy>=1.11", "xgboost>=2.0",
    "matplotlib", "joblib",
])
if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError("Drive mount did not expose /content/drive/MyDrive")
print("[setup] Drive mounted")
print("[setup] Expecting player_games.parquet and player_sample_meta.json in MyDrive/DATA")
print("[setup] Expecting schema.py in MyDrive/src")


In [ ]:
from pathlib import Path
import sys


def find_repo():
    path = Path("/content/drive/MyDrive/src")
    if not (path / "schema.py").exists():
        raise RuntimeError(
            "Upload the src package to MyDrive/src so schema.py sits at /content/drive/MyDrive/src/schema.py."
        )
    parent = path.parent
    resolved = str(parent.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    print(f"[setup] repo -> {resolved}")
    return parent


def find_data():
    path = Path("/content/drive/MyDrive/DATA")
    if not (path / "player_games.parquet").exists():
        raise RuntimeError(
            "Upload player_games.parquet to MyDrive/DATA after python -m src.player_pipeline finishes locally."
        )
    print(f"[setup] data -> {path}")
    return path


REPO = find_repo()
DATA = find_data()


## Plan

Hypothesis: eight games from the same player carry a sharper accuracy signal than one game, so validation macro MAE falls and the tails move less toward 1600.

Sweep: `SEARCH_GRID` in `src/train_quantile.py`. One axis moves at a time off the frozen point.

Selection metric: validation macro per-bin MAE of the median. Within 150, overall MAE, and coverage do not choose the row.

Baseline: a linear model on the averaged Stockfish columns, scored with the same macro MAE on validation.

Ship bar, fixed before the test score: macro MAE ≤ 120, absolute tail bias ≤ 80, typical interval width ≤ 350, and coverage ≥ 70% in every bin with at least 20 players.

Test is not an input to this plan.


In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from src.metrics import apply_interval, within_tolerance
from src.player_table import average_games
from src.schema import (
    GAMES_PER_PLAYER,
    PLAYER_BIN_COVERAGE_MIN,
    PLAYER_MACRO_MAE_MAX,
    PLAYER_TAIL_BIAS_MAX,
    PLAYER_WIDTH_MAX,
)
from src.train_quantile import (
    SEARCH_GRID,
    build_matrices,
    engine_linear_macro,
    feature_report,
    gate_sample,
    player_ship_decision,
    predict_quantiles,
    search_validation,
    summarize_predictions,
    write_bundle,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

print("[plan] selection = lowest validation macro-bin MAE; ties keep the frozen row")
print(
    f"[plan] ship if macro MAE<={PLAYER_MACRO_MAE_MAX:.0f}, "
    f"tail |bias|<={PLAYER_TAIL_BIAS_MAX:.0f}, "
    f"width<={PLAYER_WIDTH_MAX:.0f}, "
    f"bin coverage>={PLAYER_BIN_COVERAGE_MIN:.0%}"
)
for spec in SEARCH_GRID:
    print(
        f"[plan] {spec['name']:14s} depth={spec['max_depth']}  "
        f"min_child_weight={spec['min_child_weight']}  lr={spec['learning_rate']}"
    )

games = pd.read_parquet(DATA / "player_games.parquet")
meta_path = DATA / "player_sample_meta.json"
sample_meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
players = average_games(games)
short = int((players["n_games"] != GAMES_PER_PLAYER).sum())
players = players.loc[players["n_games"] == GAMES_PER_PLAYER].reset_index(drop=True)
print(f"[load] game_rows={len(games):,}  players={len(players):,}  dropped_short={short:,}")
if sample_meta:
    print(f"[load] {sample_meta.get('protocol', '')}")

splits = gate_sample(players)
for name, frame in splits.items():
    print(
        f"[split] {name:5s}  players={len(frame):,}  mean_elo={frame['elo'].mean():.1f}"
    )
bin_counts = (
    players.groupby(["elo_bin", "split"], observed=True)
    .size()
    .unstack(fill_value=0)
)
print("[bins] players per rating bin")
print(bin_counts.to_string())

built = build_matrices(splits, with_first_move=False)
print(f"[fit] features={len(built['feature_names'])}")
y_train = splits["train"]["elo"].to_numpy(dtype=float)
y_val = splits["val"]["elo"].to_numpy(dtype=float)
y_test = splits["test"]["elo"].to_numpy(dtype=float)


## Baseline

Linear regression on the averaged Stockfish block. Same validation macro MAE as the search. Test players are not used to fit the line.


In [ ]:
linear_mae = engine_linear_macro(splits["train"], splits["val"])
print(f"[baseline] engine-linear validation macro MAE={linear_mae:.2f}")


## Search

Six quantile trios. Each median early-stops on macro per-bin MAE. The row with the lowest validation macro MAE is kept. Within-150 can be higher on a row that loses.


In [ ]:
found = search_validation(
    built["matrices"]["train"],
    y_train,
    built["matrices"]["val"],
    y_val,
    splits["val"]["elo_bin"].to_numpy(),
    built["feature_names"],
)
models = found["models"]
chosen = found["chosen"]
search_table = pd.DataFrame(found["rows"])
print(search_table.round(3).to_string(index=False))
print(
    f"[search] chose {chosen['name']}  "
    f"val macro MAE={chosen['val_macro_mae']:.2f}  "
    f"engine-linear={linear_mae:.2f}  "
    f"val within 150={100 * chosen['val_within_150']:.1f}%"
)
print("[search] within-150 did not choose the row")


## Test, scored once

The scale below was chosen on the winning row's validation predictions. Re-running this cell to compare another configuration uses test as a tuning set. Run the notebook from the top instead.

`ship_ready` requires every part of the bar: macro MAE, tail bias, interval width, and per-bin coverage. A single pass on coverage is not enough.


In [ ]:
scale = float(chosen["interval_scale"])
print(
    f"[interval] scale={scale:.2f}  "
    f"validation coverage={100 * chosen['val_coverage']:.1f}%"
)

test_q = predict_quantiles(models, built["matrices"]["test"])
test_summary = summarize_predictions(
    y_test,
    splits["test"]["elo_bin"].to_numpy(),
    *test_q,
    scale,
)
decision = player_ship_decision(test_summary)
low, mid, high = apply_interval(*test_q, scale)
test_within_150 = within_tolerance(y_test, mid, 150.0)
print(
    f"[test] coverage={100 * test_summary['coverage']:.1f}%  "
    f"macro MAE={test_summary['macro_mae']:.1f}  "
    f"MAE={test_summary['mae']:.1f}  "
    f"within 150={100 * test_within_150:.1f}%  "
    f"ship_ready={decision['ship_ready']}"
)
print(f"[test] wide bins: {decision['wide_bins'] or 'none'}")
print(f"[test] tail bias bins: {decision['tail_bias_bins'] or 'none'}")
report = pd.DataFrame(test_summary["bins"])
print(report.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.scatter(y_test, mid, s=12, alpha=0.35, color="#2a6fbb")
lims = [float(min(y_test.min(), mid.min())), float(max(y_test.max(), mid.max()))]
ax.plot(lims, lims, color="black", lw=1)
ax.set_xlabel("Actual mean Elo")
ax.set_ylabel("Predicted median Elo")
ax.set_title("Test players: median vs actual Elo")
fig.tight_layout()
out_dir = DATA / "player8"
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "pred_vs_actual.png", bbox_inches="tight")
plt.show()
print("[plot] saved pred_vs_actual.png. Points above the diagonal are guessed too high.")


## What moves the guess

Importance is the rise in validation MAE when that averaged column is shuffled. The Elo curve uses validation players. Test players do not rank features.


In [ ]:
importance, top = feature_report(
    models["q50"],
    built["matrices"]["val"],
    y_val,
    n_repeats=5,
    top_k=5,
)
print("[features] top of the validation permutation ranking")
print(importance.head(12).round(3).to_string(index=False))
print()
for rank, item in enumerate(top, start=1):
    print(f"{rank}. {item['feature']}  (validation MAE rises by {item['importance']:.2f} Elo when shuffled)")
    print(f"   {item['description']}")

show = importance.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.barh(show["feature"], show["importance"], color="#415a77")
ax.set_xlabel("Validation MAE increase (Elo)")
ax.set_ylabel("Feature")
ax.set_title("Permutation importance of the eight-game median")
fig.tight_layout()
fig.savefig(out_dir / "feature_importance.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(11, 6.2))
flat = list(axes.ravel())
for axis, item in zip(flat, top):
    curve = item["curve"]
    if not curve:
        axis.set_axis_off()
        axis.set_title(item["feature"])
        continue
    xs = [point["value_mean"] for point in curve]
    ys = [point["elo_mean"] for point in curve]
    axis.plot(xs, ys, marker="o", color="#2a6fbb")
    axis.set_title(item["feature"])
    axis.set_xlabel("Feature value")
    axis.set_ylabel("Mean actual Elo")
flat[-1].set_axis_off()
fig.suptitle("Validation: mean actual Elo across each averaged feature", y=1.02)
fig.tight_layout()
fig.savefig(out_dir / "feature_curves.png", bbox_inches="tight")
plt.show()
print("[plot] saved feature_importance.png and feature_curves.png")


## Save

`model_q10.json`, `model_q50.json`, `model_q90.json`, and `model_card.json` land in `MyDrive/DATA/player8`. There is no opening encoder. The card records the ship bar and whether this run cleared it.


In [ ]:
result = {
    "chosen": chosen["name"],
    "chosen_params": {
        "max_depth": chosen["max_depth"],
        "min_child_weight": chosen["min_child_weight"],
        "learning_rate": chosen["learning_rate"],
    },
    "val_macro_mae": chosen["val_macro_mae"],
    "engine_linear_val_macro_mae": linear_mae,
    "test_coverage": test_summary["coverage"],
    "test_macro_mae": test_summary["macro_mae"],
    "test_mae": test_summary["mae"],
    "test_within_150": test_within_150,
    "ship_ready": decision["ship_ready"],
    "wide_bins": decision["wide_bins"],
    "tail_bias_bins": decision["tail_bias_bins"],
}
print("[result]", json.dumps({
    key: result[key]
    for key in ("chosen", "test_coverage", "test_macro_mae", "test_mae", "test_within_150", "ship_ready", "wide_bins", "tail_bias_bins")
}))

card = {
    "protocol": (
        "Eight games per player, features averaged, quantiles at 0.1, 0.5, 0.9. "
        "A six-row validation search selects on macro per-bin MAE. "
        "The ship bar is macro MAE, tail bias, interval width, and per-bin coverage. "
        "Test is scored once."
    ),
    "pool": "classical",
    "games_per_player": GAMES_PER_PLAYER,
    "feature_names": built["feature_names"],
    "numeric_features": built["numeric_features"],
    "search": found["rows"],
    "chosen": result["chosen_params"] | {"name": chosen["name"], "frozen": chosen["frozen"]},
    "interval_scale": scale,
    "val_coverage": chosen["val_coverage"],
    "val_macro_mae": chosen["val_macro_mae"],
    "val_mae": chosen["val_mae"],
    "val_within_150": chosen["val_within_150"],
    "test_coverage": test_summary["coverage"],
    "test_macro_mae": test_summary["macro_mae"],
    "test_mae": test_summary["mae"],
    "test_within_150": test_within_150,
    "test_bins": test_summary["bins"],
    "ship_ready": decision["ship_ready"],
    "wide_bins": decision["wide_bins"],
    "tail_bias_bins": decision["tail_bias_bins"],
    "ship_bar": {
        "macro_mae_max": PLAYER_MACRO_MAE_MAX,
        "tail_bias_max": PLAYER_TAIL_BIAS_MAX,
        "width_max": PLAYER_WIDTH_MAX,
        "bin_coverage_min": PLAYER_BIN_COVERAGE_MIN,
    },
    "engine_linear_val_macro_mae": linear_mae,
    "top_features": top,
    "watched_features": [item["feature"] for item in top],
    "sample": {
        "players": sample_meta.get("players"),
        "engine_games": sample_meta.get("engine_games"),
        "games_per_player": sample_meta.get("games_per_player"),
        "bins": sample_meta.get("bins"),
    },
}
write_bundle(out_dir, models, built["encoder"], card, importance=importance)
print(f"[save] ship_ready={decision['ship_ready']}  macro MAE={test_summary['macro_mae']:.1f}")
result


## Next steps

- If `ship_ready` is false, do not stretch the tails and do not average eight guesses from the one-game model. Those guesses share the same bias.
- The next sample is more games per player. That needs another month, because 2200–2400 is already thin at eight games.
- The page asks for eight classical games, averages the features, and shows the 10–90% band. Fewer than eight games gets no rating.
